In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [72]:
#%load_ext pyinstrument

In [73]:
USE_NEGATIVE_WGT = False
MAX_ITER = 1

In [74]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-09-06-07_36_39_PM'

In [75]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [76]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [77]:
experiment_config = {
     "experiment": {
        "model": "hill_climb",
        "exp_oof_path": "2026-08-26-05_42_22_PM_optuna_xgboost",
        "description": "xgb hill climb sample",
        "use_negative_wgt": USE_NEGATIVE_WGT,
        "max_iter": MAX_ITER
    }
}
experiment_config["experiment"]["id"] = f"{dt_str}_{experiment_config["experiment"]["model"]}"

In [78]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
y = raw_train_df[target_column]

In [79]:
exp_oof_path = Path(output_path) / "experiments" / experiment_config["experiment"]["exp_oof_path"]
tune_trials_df = pd.read_csv(exp_oof_path / "optuna_trials.csv")

trial_names = []
oof_dfs = pd.DataFrame()
oof_id = None

for subdir in exp_oof_path.iterdir():
    if not subdir.is_dir():
        continue

    trial_names.append(subdir.name)
    oof_file = subdir / "oof.csv"

    if oof_file.exists():
        file_df = pd.read_csv(oof_file)
        if oof_id is None:
            oof_id = file_df['id']
        oof_dfs[subdir.name] = file_df[target_column]

y_valid = y.iloc[oof_id]
oof_dfs

,xgboost_trial_0,xgboost_trial_1,xgboost_trial_10,xgboost_trial_11,xgboost_trial_12,xgboost_trial_13,xgboost_trial_14,xgboost_trial_15,xgboost_trial_16,xgboost_trial_17,...,xgboost_trial_45,xgboost_trial_46,xgboost_trial_47,xgboost_trial_48,xgboost_trial_49,xgboost_trial_5,xgboost_trial_6,xgboost_trial_7,xgboost_trial_8,xgboost_trial_9
0,0.468076,0.333377,0.333812,0.241975,0.262062,0.335222,0.266416,0.239746,0.240217,0.262601,...,0.253812,0.244690,0.167855,0.307484,0.285359,0.429105,0.264708,0.208057,0.448801,0.208114
1,0.961300,0.938299,0.948626,0.994228,0.991173,0.939327,0.922853,0.985203,0.986138,0.983060,...,0.979746,0.953047,0.991716,0.993844,0.993246,0.990844,0.987820,0.993877,0.956104,0.993809
2,0.999849,0.999100,0.999862,0.999967,0.999945,0.999862,0.995759,0.999943,0.999688,0.999806,...,0.998889,0.993138,0.999888,0.999967,0.999865,0.999978,0.999992,0.999998,0.999751,0.999991
3,0.994608,0.986703,0.990228,0.999668,0.999108,0.996028,0.971993,0.999213,0.995884,0.999491,...,0.994239,0.978917,0.999083,0.999552,0.998959,0.999860,0.999480,0.999967,0.995357,0.999904
4,0.382653,0.453041,0.443854,0.409762,0.584378,0.466554,0.565250,0.401545,0.486953,0.674238,...,0.499460,0.491967,0.493443,0.420282,0.468878,0.222053,0.573161,0.516801,0.541556,0.408385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138269,0.998143,0.991814,0.997811,0.999606,0.998901,0.996317,0.976595,0.996978,0.998118,0.997533,...,0.995473,0.981229,0.998660,0.999451,0.999768,0.999586,0.998455,0.999493,0.998052,0.999637
138270,0.996626,0.995381,0.995213,0.996545,0.998869,0.995798,0.983228,0.997169,0.997045,0.995874,...,0.991673,0.985175,0.990737,0.999115,0.998804,0.998298,0.991240,0.998695,0.996066,0.997833
138271,0.069462,0.218815,0.236040,0.037729,0.072827,0.238128,0.129251,0.106701,0.083388,0.093966,...,0.097958,0.105190,0.049580,0.025594,0.113789,0.077918,0.074791,0.085888,0.072037,0.042195
138272,0.234323,0.321914,0.327710,0.314199,0.257322,0.276815,0.454227,0.249434,0.239110,0.167609,...,0.322147,0.357163,0.289625,0.245529,0.386222,0.403365,0.284865,0.238811,0.178831,0.158689


In [80]:
best_start_model_idx = tune_trials_df['value'].idxmax()
ensemble_score = tune_trials_df.iloc[best_start_model_idx]['value']
ensemble_oof = oof_dfs[f'xgboost_trial_{best_start_model_idx}'].to_numpy().reshape(-1, 1)
ensemble_oof

array([[0.26470792],
       [0.98782027],
       [0.9999925 ],
       ...,
       [0.07479139],
       [0.28486517],
       [0.23614253]], shape=(138274, 1))

In [81]:
start = -0.50
if not USE_NEGATIVE_WGT: 
    start = 0.01

weights = np.arange(start, 0.51, 0.01)
weights.shape

(50,)

In [82]:
y_valid_np = np.asarray(y_valid)
pos_mask = y_valid_np == 1

n_pos = pos_mask.sum()
n_neg = len(y_valid_np) - n_pos

def fast_auc_batch(y_true, predictions):
    ranks = rankdata(predictions, axis=0, method="average")
    pos_rank_sum = ranks[y_true == 1].sum(axis=0)
    auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

    return auc

In [83]:
iter = 0

models = [f'xgboost_trial_{best_start_model_idx}']
wgts = []
scores = [ensemble_score]

ensemble_oof = oof_dfs[models[0]].to_numpy().reshape(-1, 1)

start_time = time.time()

while iter < MAX_ITER:
    model_valid_scores = []
    best_weight_idxs = []

    for model in oof_dfs.columns:
        model_oof = oof_dfs[model].to_numpy().reshape(-1, 1)

        weight_oofs = (
            model_oof * weights
            + ensemble_oof * (1 - weights)
        )

        weight_scores = fast_auc_batch(
            y_valid_np,
            weight_oofs
        )

        max_idx = np.argmax(weight_scores)
        max_weight_score = weight_scores[max_idx]

        model_valid_scores.append(max_weight_score)
        best_weight_idxs.append(max_idx)

    model_valid_scores = np.array(model_valid_scores)

    max_model_idx = np.argmax(model_valid_scores)
    max_weight_idx = best_weight_idxs[max_model_idx]

    candidate_score = model_valid_scores[max_model_idx]

    model_name = oof_dfs.columns[max_model_idx]
    candidate_weight = weights[max_weight_idx]

    candidate_oof = oof_dfs[model_name].to_numpy().reshape(-1, 1)

    if candidate_score <= ensemble_score:
        break

    ensemble_oof = (
        candidate_oof * candidate_weight
        + ensemble_oof * (1 - candidate_weight)
    )

    ensemble_score = candidate_score

    models.append(model_name)
    wgts.append(candidate_weight)
    scores.append(candidate_score)

    iter += 1

    wgt = np.array([1.0])

    for w in wgts:
        wgt = wgt * (1 - w)
        wgt = np.concatenate([wgt, np.array([w])])

    model_weight_df = pd.DataFrame({
        'model': models,
        'weight': wgt,
        'ensemble_score': scores
    })
    print(model_weight_df)

elapsed = time.time() - start_time

              model  weight  ensemble_score
0   xgboost_trial_6    0.54        0.966998
1  xgboost_trial_11    0.46        0.967419


In [84]:
wgt = np.array([1.0])

for w in wgts:
    wgt = wgt * (1 - w)
    wgt = np.concatenate([wgt, np.array([w])])

In [85]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "exp_oof_path": experiment_config["experiment"]["exp_oof_path"],
    "use_negative_wgt": USE_NEGATIVE_WGT,
    "max_iter": MAX_ITER,
    "primary_metric": {
        "name": "auc",
        "value": round(scores[-1], 5)
    },
    "models": models,
    "weights": wgt.tolist(),
    "oof_scores": scores,
    "validation": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [86]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

model_weight_df.to_csv(experiment_path / "weights.csv", index=False)